## Stochastic Gradient Descent

In Stochastic Gradient Descent, we choose a learning rate such as 1. We then subtract learning_rate * parameter_gradients from the actual parameter values.

In [ ]:
class Optimizer_SGD:
    def __init__(self, learning_rate = 1.0):
        self.learning_rate = learning_rate

    #update parameters
    def update_params(self, layer):
        layer.weights += -self.learning_rate * layer.dweights
        layer.biases += -self.learning_rate * layer.dbiases

In [ ]:
import numpy as np
from nnfs.datasets import vertical_data, spiral_data
import nnfs

In [ ]:
class Dense_layer:

    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    # we want to remember our input to calculate gradient of weights
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    
    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis = 0, keepdims = True)
        self.dinputs = np.dot(dvalues, self.weights.T)


class Activation_Relu:

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0


class Activation_softmax:
    
    def forward(self, inputs):

        exp_values = np.exp(inputs - np.max(inputs, axis = 1, keepdims = True))
        probabilities = exp_values/np.sum(exp_values, axis = 1, keepdims = True)
        self.output = probabilities

class Loss():
    def calculate(self, output, y):
        #calculate sample losses 
        sample_losses = self.forward(output, y)

        data_loss = np.mean(sample_losses)

        return data_loss
    
class Loss_CategoricalCrossentropy(Loss):

    def forward(self, y_pred, y_true):

        samples = len(y_pred)

        #We are capping the y_pred for preventing log(0) or log(high value) both are not defined
        y_pred_clipped = np.clip(y_pred, 1e-7, 1- 1e-7)

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]

        if len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis = 1)

        negative_log_likelihood = -np.log(correct_confidences)
        return negative_log_likelihood

class Loss_CategoricalCrossentropy(Loss):

    def forward(self, y_pred, y_true):

        samples = len(y_pred)

        #We are capping the y_pred for preventing log(0) or log(high value) both are not defined
        y_pred_clipped = np.clip(y_pred, 1e-7, 1- 1e-7)

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]

        if len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis = 1)

        negative_log_likelihood = -np.log(correct_confidences)
        return negative_log_likelihood
    
    def backward(self, dvalues, y_true):
        samples = len(dvalues) # Number of samples

        #Number of labels in every sample
        #We'll use the first sample to count them
        labels = len(dvalues[0])

        #If labels are sparse then, turn then into one-hot vector
        if len(y_true.shape) == 1:
            y_true =   np.eye(labels)[y_true]

        #calculate gradient
        self.dinputs = -y_true / dvalues

        #Normalize gradient
        self.dinputs = self.dinputs / samples

class Activation_softmax:
    
    def forward(self, inputs):

        exp_values = np.exp(inputs - np.max(inputs, axis = 1, keepdims = True))
        probabilities = exp_values/np.sum(exp_values, axis = 1, keepdims = True)
        self.output = probabilities

    def backward(self, dvalues):
        self.dinputs = np.empty_like(dvalues)

        for index, (single_output, single_dvalues) in enumerate(zip(self.output, dvalues)):
            single_output = single_output.reshape(-1,1)

            jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T)

            #calculate sample-wise gradient
            #and add it to the array of sample gradients
            self.dinputs[index] = np.dot(jacobian_matrix, single_dvalues)

class Activation_Softmax_Loss_CategoricalCrossentropy():

    def __init__(self):
        self.activation = Activation_softmax()
        self.loss = Loss_CategoricalCrossentropy()

    # Forward pass
    def forward(self, inputs, y_true):
        #output layer activation function
        self.activation.forward(inputs)

        #set the output
        self.output = self.activation.output

        #calculate and return loss value
        return self.loss.calculate(self.output, y_true)
    
    def backward(self, dvalues, y_true):

        samples = len(dvalues)

        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis = 1)

        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        self.dinputs = self.dinputs/samples


In [ ]:
X, y = vertical_data(samples = 100, classes = 3)

# Dense layer with 2 input and 64 output values (64 neurons).
dense1 = Dense_layer(2,64)

activation1 = Activation_Relu()

#Create second dense layer that take output from previous layer hence 3 inputs and 3 output values
dense2 = Dense_layer(64,3)

#We are now using new class to calculate final softmax activation and loss calculation.
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

#Create optimizer
optimizer = Optimizer_SGD()

# Perform a forward pass of our training data through this layer
dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)

loss = loss_activation.forward(dense2.output, y)
print(loss)

In [ ]:
predictions = np.argmax(loss_activation.output, axis = 1)
if len(y.shape) == 2:
    y = np.argmax(y, axis = 1)
accuracy = np.mean(predictions == y)
print(accuracy)

In [ ]:
## Backpropogation

loss_activation.backward(loss_activation.output, y)
dense2.backward(loss_activation.dinputs)
activation1.backward(dense2.dinputs)
dense1.backward(activation1.dinputs)

In [ ]:
optimizer.update_params(dense1)
optimizer.update_params(dense2)

We can perform optimization many times using loop. We can optimize until a stopping condition is met.
Each full pass through all training data is called an epoch.

In [ ]:
for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    loss = loss_activation.forward(dense2.output, y)
    predictions = np.argmax(loss_activation.output, axis = 1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f}')
        
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.update_params(dense1)
    optimizer.update_params(dense2)


* Our accuracy got stuck at 93% and then at 95%. This is pretty good but we don't know if it can get better. 
Sometimes the accuracy doesn't improve because we get stuck in local minimum. This usually happens because of low learning rate.
* One way to check how far are we from global minimum is to look at how far is our loss value from 0.
* The model can get stuck in "not so deeper" local minimum as well. The model follows the direction of steepest descent of loss function, no matter how large or slight the descent is.
* Setting a learning too high can cause the model to overshoot the global minimum even if it reaches there. It can also cause gradient explosion.

## Gradient Exlosion

A gradient explosion is a situation where the parameter updates cause the function's output to rise instead of fall and with each step, the loss value and gradient becomes larger. At some point, the floating point variable limitation cause an overflow and it cannot hold value of this size.

## Learning Rate Decay

We start with a large learning rate and then decrease it during training based on the value of loss. Another option is to program a Decay rate, which steadily decays the learning rate per batch or epoch.

### Exponential Decay
We update the learning rate each step by the reciprocal of the step count fraction. This fraction will be a new hyperparameter, called learning rate decay.

In [ ]:
starting_learning_rate = 1
learning_rate_decay = 0.1
step = 1

learning_rate = starting_learning_rate * (1/(1+learning_rate_decay * step))
print(learning_rate)

In [ ]:
step = 20
learning_rate = starting_learning_rate * (1/(1+learning_rate_decay * step))
print(learning_rate)

In [ ]:
class Optimizer_SGD:
    def __init__(self, learning_rate = 1.0, decay = 0):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1/(1+ self.decay * self.iterations))


    #update parameters
    def update_params(self, layer):
        layer.weights += -self.current_learning_rate * layer.dweights
        layer.biases += -self.current_learning_rate * layer.dbiases

    def post_update_params(self):
        self.iterations += 1


In [ ]:
optimizer = Optimizer_SGD(decay = 1e-2)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    loss = loss_activation.forward(dense2.output, y)
    predictions = np.argmax(loss_activation.output, axis = 1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f},' +
              f'lr: {optimizer.current_learning_rate}')
        
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

It didn't solve the problem. The model seems to get stuck a lot, probably as the decay was high and learning rate decreased a little too quicly and the model got stuck in local minimum.

In [ ]:
optimizer = Optimizer_SGD(decay = 1e-3)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    loss = loss_activation.forward(dense2.output, y)
    predictions = np.argmax(loss_activation.output, axis = 1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f},' +
              f'lr: {optimizer.current_learning_rate}')
        
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

## Stochastic Gradient Descent with Momentum

Momentum creates a rolling average of gradients over some number of updates and uses this average with the unique gradient at each step. The gradient point towards the current steepest loss ascent for that step which may not follow descent towards the global minimum. 

Momentum is defined as a value between 0 and 1, representing the fraction of the previous parameter update to retain, and subtracting our actual gradient, multiplied by the learning rate, from it.

In [ ]:
class Optimizer_SGD:
    def __init__(self, learning_rate = 1.0, decay = 0, momentum = 0):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.momentum = momentum

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1/(1+ self.decay * self.iterations))


    #update parameters
    def update_params(self, layer):

        if self.momentum:
            #if layer does not contain momentum arrays, create them filled with zeros
            if not hasattr(layer, 'weight_momentums'):
                layer.weight_momentums = np.zeros_like(layer.weights)
                layer.bias_momentums = np.zeros_like(layer.biases)

            weight_updates = self.momentum * layer.weight_momentums - self.current_learning_rate * layer.dweights
            layer.weight_momentums = weight_updates

            bias_updates = self.momentum * layer.bias_momentums - self.current_learning_rate * layer.dbiases
            layer.weight_momentums = bias_updates

        else:
            weight_updates = - self.current_learning_rate * layer.dweights

            bias_updates = - self.current_learning_rate * layer.dbiases

        layer.weights += weight_updates
        layer.biases += bias_updates

    def post_update_params(self):
        self.iterations += 1

In [ ]:
optimizer = Optimizer_SGD(decay = 1e-3, momentum = 0.5)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    loss = loss_activation.forward(dense2.output, y)
    predictions = np.argmax(loss_activation.output, axis = 1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f},' +
              f'lr: {optimizer.current_learning_rate}')
        
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

In [ ]:
optimizer = Optimizer_SGD(decay = 1e-3, momentum = 0.9)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    loss = loss_activation.forward(dense2.output, y)
    predictions = np.argmax(loss_activation.output, axis = 1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(f'epoch: {epoch}, ' +
              f'acc: {accuracy:.3f}, ' +
              f'loss: {loss:.3f},' +
              f'lr: {optimizer.current_learning_rate}')
        
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()